# Predict


In [1]:
import geopandas as gpd
import rasterio as rio
import numpy as np

import xdem
import geoutils as gu
import subkart
import importlib
import joblib


In [2]:
gdf = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/Basisdata_15_More_og_Romsdal_25833_Dybdedata_Dybdeareal.geo.parquet")


In [3]:
marine_vanntyper = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/mdir/NyTypologi2022.geo.parquet").to_crs(gdf.crs)

## Predict for AOI

In [4]:
gdf_bunn_moere = gpd.read_file("https://storage.googleapis.com/niva-geodata/MarintNaturKart/bunn_in_kommuner_sea.geojson")
gdf_clip = gdf_bunn_moere.to_crs(gdf.crs)

vec_clip = gu.Vector(gdf_clip)

In [6]:

clipped_marine_vanntyper = gdf_clip.overlay(marine_vanntyper)

In [7]:
clipped_gdf = gdf_clip.overlay(gdf)

depth, slope, one_hot_types = subkart.features.build_basis_raster(clipped_gdf, clipped_marine_vanntyper)

features = np.concatenate([np.stack([depth.data.data, slope.data.data], axis=-1), one_hot_types], axis=-1)

valid_attrs = (~np.isnan(depth.data.data)) & (~np.isnan(slope.data.data))

X = features[valid_attrs]

/home/kim/work/marint-naturkart-nivaR/.venv/lib/python3.11/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(
/home/kim/work/marint-naturkart-nivaR/.venv/lib/python3.11/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


In [8]:
classifier = joblib.load("../data_generated/classifier.joblib")

In [9]:
Y_pred = classifier.predict(X)

In [10]:
pred_map = np.full(depth.data.shape[-2:], np.nan, dtype=np.float32)
pred_map[valid_attrs] = Y_pred.astype(np.float32)

In [11]:

pred_raster = gu.Raster.from_array(pred_map, transform=depth.transform, crs=depth.crs)

/home/kim/work/marint-naturkart-nivaR/.venv/lib/python3.11/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


# Vectorize output

In [12]:
pred_vec = pred_raster.polygonize()

/home/kim/work/marint-naturkart-nivaR/.venv/lib/python3.11/site-packages/geoutils/interface/raster_vector.py:102: RuntimeWarning: invalid value encountered in cast
  shapes(source_raster.data.astype(final_dtype), mask=bool_msk, transform=source_raster.transform)


In [13]:
# Map raster_value (0/1) to BunnType using existing mapping_values
reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
pred_vec["BunnType"] = pred_vec["raster_value"].map(reverse_map)

pred_vec["BunnType"]

0       fastbunn
1       fastbunn
2       fastbunn
3       fastbunn
4       fastbunn
          ...   
7534    fastbunn
7535    fastbunn
7536     løsbunn
7537     løsbunn
7538    fastbunn
Name: BunnType, Length: 7539, dtype: object

In [14]:
gdf_final = pred_vec.ds.dissolve(by="BunnType", as_index=False, method="coverage")

In [15]:
gdf_final = gdf_final.to_crs("EPSG:25833")

In [16]:
fname = subkart.utils.to_filename(f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "moere-og-romsdal", "latest", gdf_final.crs.to_epsg())
gdf_final = gdf_final.drop(columns=["depth_range", "class"], errors="ignore")
gdf_final.to_file(f"{fname}.geojson", driver="GeoJSON")

In [ ]:
gdf_final.to_file(f"{fname}.gpkg", layer="soft_hard_bottom", driver="GPKG")